# Crop Yield Model Analysis

This notebook summarizes the cleaning, feature engineering, and model evaluation workflow for the Team Beso challenge submission.

In [ ]:
import pandas as pd
from pathlib import Path
from src.cleaning import clean_plot_data, clean_weather_data, clean_market_price
from src.features import add_weather_features, attach_price_analysis

root = Path('..')
train = pd.read_csv(root / 'data' / 'raw' / 'crop_yield_train.csv')
test = pd.read_csv(root / 'data' / 'raw' / 'crop_yield_leaderboard_test.csv')
weather = pd.read_csv(root / 'data' / 'raw' / 'regional_weather.csv')
price = pd.read_csv(root / 'data' / 'raw' / 'market_prices.csv')

train_clean, test_clean, _ = clean_plot_data(train, test)
weather_clean, _ = clean_weather_data(weather)
price_clean, _ = clean_market_price(price)

train_features = add_weather_features(train_clean, weather_clean)
train_master = attach_price_analysis(train_features, price_clean)
print(train_master.shape)
print(train_master[['season_mean_temp', 'season_rainfall_total']].head())

## Model training notes

The final model uses a RandomForest regressor trained with cleaned agronomic inputs and engineered seasonal weather features. The validation configuration is a train/validation split, avoiding any leakage from the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from src.train import build_preprocessor, build_feature_columns

feature_cols = build_feature_columns()
X = train_master[feature_cols].copy()
y = train_master['yield_tons_per_ha'].copy()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
pre = build_preprocessor(X_train)
model = RandomForestRegressor(random_state=42, n_estimators=150, max_depth=12, n_jobs=-1, min_samples_leaf=1)
pipe = Pipeline([('prep', pre), ('model', model)])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_val)
print({
    'rmse': mean_squared_error(y_val, pred, squared=False),
    'mae': mean_absolute_error(y_val, pred),
    'r2': r2_score(y_val, pred)
})